# 01 · Ingest
Sensor CSV → bronze (Parquet, values unchanged, verified) → **one fixed-step timeline** for machines 3 and 4 with sensors, manual status and emergency events. Maintenance history goes to its own table because it has no `machine_id`.

In [ ]:
import sys, json, shutil, time
from pathlib import Path
from datetime import datetime
ROOT = Path.cwd() if (Path.cwd() / "config.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import duckdb, pandas as pd
import pyarrow.parquet as pq
import config as C

SRC_CSV  = Path(r"E:\analisis_mesin\machine_sensors.csv")   # long format, no header
SRC_DIR  = Path(r"E:\analisis_mesin")                       # machine_util, emergencies, maintenance_*
MACHINES = [3, 4]
RES      = 5                                                # seconds per slot (see section 3)

SENS = C.BRONZE / "sensors"
for p in (C.BRONZE, C.REF, C.SILVER):
    p.mkdir(parents=True, exist_ok=True)
con = duckdb.connect()
con.execute(f"SET memory_limit='3GB'; SET threads=2; SET preserve_insertion_order=false; SET enable_progress_bar=false; SET temp_directory='{C.DUCKDB_TEMP}'")
q = lambda p: Path(p).as_posix()
CSV = (f"read_csv('{q(SRC_CSV)}', header=false, columns={{'machine_id':'INTEGER','param':'VARCHAR',"
       "'value':'DOUBLE','ts':'TIMESTAMP','ts2':'TIMESTAMP'})")
BR = f"read_parquet('{q(SENS)}/**/*.parquet', hive_partitioning=true)"

## 1 · CSV → bronze
Partitioned by machine and month. Written to a temporary folder and renamed when done, so an interrupted run never leaves a half-written bronze.

In [ ]:
if not SENS.exists():
    tmp = SENS.with_name("sensors_WRITING")
    shutil.rmtree(tmp, ignore_errors=True)
    t0 = time.time()
    con.execute(f"COPY (SELECT *, strftime(ts, '%Y-%m') AS month FROM {CSV}) TO '{q(tmp)}' "
                "(FORMAT PARQUET, PARTITION_BY (machine_id, month), COMPRESSION ZSTD)")
    tmp.rename(SENS)
    print(f"Conversion took {(time.time() - t0) / 60:.1f} min")
print(f"Patch {C.PATCH} | Bronze {sum(f.stat().st_size for f in SENS.rglob('*.parquet')) / 1e9:.2f} GB, {len(list(SENS.glob('*/*')))} partitions")

## 2 · Verify bronze = CSV
Per machine and parameter: row count, min/max time and value sum (relative tolerance 1e-9).

In [ ]:
AGG = "machine_id, param, COUNT(*) n, MIN(ts) t0, MAX(ts) t1, SUM(value) s"
a = con.sql(f"SELECT {AGG} FROM {CSV} GROUP BY ALL").df()
b = con.sql(f"SELECT {AGG} FROM {BR} GROUP BY ALL").df().astype({"machine_id": int})
chk = a.merge(b, on=["machine_id", "param"], how="outer", suffixes=("_csv", "_br"))
chk["ok"] = ((chk.n_csv == chk.n_br) & (chk.t0_csv == chk.t0_br) & (chk.t1_csv == chk.t1_br)
             & ((chk.s_csv - chk.s_br).abs() <= 1e-9 * chk.s_csv.abs() + 1e-6))
PASSED = bool(chk.ok.all())
chk.rename(columns={"machine_id": "mesin", "n_csv": "baris_csv", "n_br": "baris_bronze", "t0_csv": "awal_csv",
                    "t0_br": "awal_bronze", "t1_csv": "akhir_csv", "t1_br": "akhir_bronze", "s_csv": "jumlah_csv",
                    "s_br": "jumlah_bronze", "ok": "lolos"}).to_csv(C.TAB / "ingest_verifikasi.csv", index=False)
print(f"Rows CSV {int(chk.n_csv.sum()):,} | bronze {int(chk.n_br.sum()):,} ->", "PASSED" if PASSED else "FAILED")
display(chk[~chk.ok])

## 3 · Per-slot summary and resolution
Per slot: electrical = mean, vibration = max, `w` = last value, `n_ir` = number of current samples. Processed one partition (machine, month) at a time to keep memory low.
The grid cannot be finer than the sensor rate: if mean `n_ir` per slot is well below 1, most slots are empty. Change `RES` and rerun from here if needed.

In [ ]:
prof = con.sql(f"SELECT machine_id, param, COUNT(*) n, MIN(value) min, AVG(value) mean, MAX(value) max "
               f"FROM {BR} GROUP BY ALL ORDER BY ALL").df()
prof.rename(columns={"machine_id": "mesin", "n": "jumlah", "mean": "rata"}).to_csv(C.TAB / "ingest_profil.csv", index=False)
have = set(prof.param)
cols = [f"avg(value) FILTER (param = '{p}') AS {'i_s' if p == 'is' else p}"
        for p in ["ir", "is", "it", "va", "vAR", "freq", "vrn", "vsn", "vtn", "vrs", "vst", "vrt", "temperature"] if p in have]
cols += [f"max(value) FILTER (param = '{a}_axis_speed') AS v{a}" for a in "xyz"]
cols += ["arg_max(value, ts) FILTER (param = 'w') AS w", "count(*) FILTER (param = 'ir') AS n_ir"]

SLOT = C.INTERIM / "slot"
shutil.rmtree(SLOT, ignore_errors=True); SLOT.mkdir(parents=True)
for f in sorted(SENS.glob("*/*")):
    m, mon = int(f.parent.name.split("=")[1]), f.name.split("=")[1]
    con.execute(f"COPY (SELECT {m} AS machine, time_bucket(INTERVAL '{RES} seconds', ts) AS t, {', '.join(cols)} "
                f"FROM read_parquet('{q(f)}/*.parquet') GROUP BY ALL) TO '{q(SLOT / f'm{m}_{mon}.parquet')}' (FORMAT PARQUET)")
    print("done:", f.parent.name, f.name)
con.execute(f"CREATE OR REPLACE VIEW s AS SELECT * FROM read_parquet('{q(SLOT)}/*.parquet')")
display(con.sql("SELECT machine, COUNT(*) slots_with_data, AVG(n_ir) n_ir_mean, quantile_cont(n_ir, 0.1) n_ir_p10 "
                "FROM s GROUP BY 1 ORDER BY 1").df().round(2))

## 4 · Reference files and events
- `machine_util`, `emergencies`, `maintenance_*` are copied unchanged to the ref folder.
- Emergencies are logged in **local time (WIB)**, converted to UTC (−7 h) and attached to the slot of the button press. Several presses in one slot are merged.
- Maintenance history: `created_at` (UTC) is the end time, start = end − `down_time_duration` (ms). **Not linked to a machine yet** (no work-order table).
- Records on test days or by commissioning staff (`C.COMMISSIONING_STAFF`) are kept but marked `test` (maintenance) or `(test)` (emergency) so later notebooks can exclude them.

In [ ]:
for f in ["machine_util.csv", "emergencies.csv", "maintenance_histories.csv", "maintenance_schedules.csv"]:
    shutil.copy2(SRC_DIR / f, C.REF / f)
read = lambda f: pd.read_csv(C.REF / f, na_values=["NULL"], encoding_errors="replace")

em = read("emergencies.csv")
em["t"] = (pd.to_datetime(em.triggered_at) - pd.Timedelta(hours=7)).dt.floor(f"{RES}s")
em["note"] = "ack " + em.acknowledged_at.fillna("-").astype(str).str[:19] + " WIB"
ev = (em.groupby(["machine_id", "t"]).agg(n=("id", "size"), description=("note", "; ".join))
        .reset_index().rename(columns={"machine_id": "machine"}))
ev["event"] = "EMERGENCY x" + ev.n.astype(str)
test_em = em.acknowledged_by.str.upper().isin(C.COMMISSIONING_STAFF).groupby([em.machine_id, em.t]).any().to_numpy()
ev.loc[test_em, "event"] += " (test)"
con.register("ev", ev[["machine", "t", "event", "description"]])

mh = read("maintenance_histories.csv")
mh["end"] = pd.to_datetime(mh.created_at)
mh["start"] = mh.end - pd.to_timedelta(mh.down_time_duration, unit="ms")
mh["duration_h"] = mh.down_time_duration / 3.6e6
mh["machine"] = pd.NA                                    # filled in notebook 04 (inferred + flagged)
mh["test"] = ((mh.start + pd.Timedelta(hours=7)).dt.strftime("%Y-%m-%d").isin(C.TEST_DAYS)
              | mh.performed_by.str.upper().isin(C.COMMISSIONING_STAFF))   # kept, but excluded from MTBF
mh = mh[["start", "end", "duration_h", "machine", "test", "work_order_id", "summary", "result",
         "is_finished", "performed_by", "inspect_by"]].sort_values("start")
mh.to_parquet(C.SILVER / "maintenance.parquet", index=False)
display(mh.drop(columns="summary").round({"duration_h": 2}))

## 5 · Combined timeline
Fixed `RES`-second grid for both machines from the first to the last sample. Slots without data are kept (`has_data = False`) so gaps stay visible.
Status = last status at or before the slot (ASOF join, UTC). $I = (I_R + I_S + I_T)/3$. `meter_off` = data present but frequency missing or < 1 Hz. Built per machine, then streamed into one file.

In [ ]:
UTIL = (f"read_csv('{q(C.REF / 'machine_util.csv')}', header=false, "
        "columns={'machine_id':'INTEGER','operator':'VARCHAR','status':'VARCHAR','ts':'TIMESTAMP'})")
OUT = C.SILVER / "timeline.parquet"
t0, t1 = con.execute("SELECT MIN(t), MAX(t) FROM s").fetchone()
parts = []
for m in MACHINES:
    parts.append(C.INTERIM / f"timeline_m{m}.parquet")
    con.execute(f'''COPY (
        SELECT g.machine, g.t, g.t + INTERVAL 7 HOUR AS t_local, s.* EXCLUDE (machine, t),
               s.n_ir IS NOT NULL AS has_data,
               (s.ir + s.i_s + s.it) / 3 AS I,
               s.n_ir IS NOT NULL AND (s.freq IS NULL OR s.freq < 1) AS meter_off,
               u.status, u.operator, ev.event, ev.description
        FROM (SELECT {m} AS machine, generate_series AS t
              FROM generate_series(TIMESTAMP '{t0}', TIMESTAMP '{t1}', INTERVAL '{RES} seconds')) g
        LEFT JOIN (SELECT * FROM s WHERE machine = {m}) s USING (machine, t)
        ASOF LEFT JOIN (SELECT * FROM {UTIL} WHERE machine_id = {m}) u ON g.t >= u.ts
        LEFT JOIN (SELECT * FROM ev WHERE machine = {m}) ev ON ev.t = g.t
        ORDER BY g.t
    ) TO '{q(parts[-1])}' (FORMAT PARQUET)''')
    print("timeline machine", m, "done")
with pq.ParquetWriter(OUT, pq.read_schema(parts[0]), compression="zstd") as w:   # stream, never all in RAM
    for p in parts:
        for batch in pq.ParquetFile(p).iter_batches():
            w.write_batch(batch)

summ = con.sql(f'''SELECT machine, COUNT(*) AS slots, MIN(t) AS first_t, MAX(t) AS last_t,
    ROUND(100 * AVG(has_data::INT), 2) AS pct_has_data, SUM(n_ir) AS ir_samples, COUNT(event) AS emergency_slots
    FROM '{q(OUT)}' GROUP BY 1 ORDER BY 1''').df()
summ["ir_matches_bronze"] = [summ.ir_samples[i] == chk.query("machine_id == @m and param == 'ir'").n_br.sum()
                             for i, m in enumerate(summ.machine)]
display(summ)
print("Emergencies outside sensor range:", int((~em.t.between(summ.first_t.min(), summ.last_t.max())).sum()), "of", len(em))
print(f"{OUT}  ({OUT.stat().st_size / 1e6:.0f} MB)")

## 6 · Manifest
One file per run in `data/manifest/`, never overwritten.

In [ ]:
man = {"patch": C.PATCH, "time": datetime.now().isoformat(timespec="seconds"), "duckdb": duckdb.__version__,
       "res_s": RES, "sources": {str(p): p.stat().st_size for p in [SRC_CSV, *(SRC_DIR / f for f in
                  ["machine_util.csv", "emergencies.csv", "maintenance_histories.csv", "maintenance_schedules.csv"])]},
       "verification_passed": PASSED, "sensor_rows": int(chk.n_csv.sum()), "timeline": summ.astype(str).to_dict("records")}
d = C.ROOT / "data" / "manifest"; d.mkdir(exist_ok=True)
(d / f"{C.PATCH}_{datetime.now():%Y%m%d-%H%M}.json").write_text(json.dumps(man, indent=1), encoding="utf-8")
print(json.dumps({k: man[k] for k in ("verification_passed", "sensor_rows", "res_s")}, indent=1))